In [4]:
import json
import h5py
import pandas as pd
import numpy as np
from scipy.spatial.transform import Rotation

In [13]:
import json
import h5py
import pandas as pd
import numpy as np
from scipy.spatial.transform import Rotation

# 1. Leer el archivo de especificaciones de sensores.
with open("../../primer_modelo/Aventa_sensors.json", "r") as f:
    sensors_specs = json.load(f)

# 2. Leer el archivo HDF5 y organizarlo en un diccionario.
# Se verifica si el archivo tiene un grupo "Aventa" en la raíz; en ese caso se usa ese grupo.
datasets = {}
with h5py.File("../../primer_modelo/Aventa_Taggenberg_06_02_2022.hdf5", "r") as hdf:
    if "Aventa" in hdf:
        group = hdf["Aventa"]
    else:
        group = hdf
    data_aventa = {}
    for channel in group.keys():
        if channel == "ChannelList":
            continue

        data_obj = group[channel]
        # Si es un dataset, lo leemos directamente
        if isinstance(data_obj, h5py.Dataset):
            data = data_obj[...]
        # Si es un grupo, se toma el primer dataset disponible dentro del grupo.
        elif isinstance(data_obj, h5py.Group):
            subkeys = list(data_obj.keys())
            if not subkeys:
                print(f"No se encontró un dataset en el grupo {channel}.")
                continue
            sub_obj = data_obj[subkeys[0]]
            if isinstance(sub_obj, h5py.Dataset):
                data = sub_obj[...]
            else:
                print(f"No se encontró un dataset en el grupo {channel}.")
                continue
        else:
            continue

        n_samples = len(data)
        # Vector de tiempo sintético: 0, 1, 2, ... n_samples-1
        time_vec = np.arange(n_samples)
        data_aventa[channel] = {"Time": time_vec, "Value": data}
    datasets["Aventa"] = data_aventa

def rotate_sensor_to_tower(sensor, dataset):
    """
    Rota los datos de un sensor al marco de referencia de la torre.

    Parámetros:
      sensor: diccionario con la especificación del sensor (debe incluir 'channels' y 'yaw-pitch-roll').
      dataset: diccionario con los datos medidos, donde las claves son los nombres de los canales.

    Retorna:
      DataFrame con las columnas "Time", "Xt", "Yt" y "Zt".
      Si falta información (canales incompletos o ángulos no definidos), se omite la rotación y se retorna None.
    """
    channels = sensor.get('channels', [])
    sensor_id = sensor.get('sensor_placement', {}).get('id', 'desconocido')
    
    # Verificar que existan al menos 3 canales
    if len(channels) < 3:
        print(f"Sensor {sensor_id} no tiene suficientes canales.")
        return None
    
    # Verificar que cada canal esté definido y exista en el dataset
    for ch in channels:
        if ch == "None":
            print(f"El canal '{ch}' está definido como 'None' para sensor {sensor_id}. Se omite la rotación.")
            return None
        if ch not in dataset:
            print(f"Error: el canal '{ch}' no se encontró en el dataset para sensor {sensor_id}.")
            return None

    # Verificar que el sensor tenga 'yaw-pitch-roll' definido y completo
    sp = sensor.get('sensor_placement', {})
    if 'yaw-pitch-roll' not in sp:
        print(f"Sensor {sensor_id} no tiene 'yaw-pitch-roll' definido; se omite la rotación.")
        return None
    ypr = sp['yaw-pitch-roll']
    if any(val is None or str(val) == "None" for val in ypr):
        print(f"Sensor {sensor_id} tiene ángulos no definidos en 'yaw-pitch-roll'; se omite la rotación.")
        return None

    # Crear el objeto de rotación con la convención 'ZYX'
    rotation = Rotation.from_euler('ZYX', ypr, degrees=True)
    
    try:
        X_data = dataset[channels[0]]["Value"]
        Y_data = dataset[channels[1]]["Value"]
        Z_data = dataset[channels[2]]["Value"]
        # Usamos el vector de tiempo del primer canal
        time_array = dataset[channels[0]]["Time"]
    except KeyError as e:
        print(f"Error al leer canales para sensor {sensor_id}: {e}")
        return None

    # Construir el DataFrame con los datos originales
    df = pd.DataFrame({
        "Time": time_array,
        "Xs": X_data,
        "Ys": Y_data,
        "Zs": Z_data
    })
    
    # Aplicar la rotación a los datos locales
    data_local = df[['Xs', 'Ys', 'Zs']].to_numpy()
    data_rotated = rotation.apply(data_local)
    
    # Construir el DataFrame final (sólo con la columna de tiempo y los datos rotados)
    df_rotated = pd.DataFrame({
        "Time": df["Time"],
        "Xt": data_rotated[:, 0],
        "Yt": data_rotated[:, 1],
        "Zt": data_rotated[:, 2]
    })
    
    return df_rotated

# 3. Iterar sobre los sensores que no pertenecen al componente 'tower' y aplicar la rotación.
rotated_dataframes = {}

for sensor in sensors_specs['sensors']:
    sp = sensor.get('sensor_placement', {})
    component = sp.get('wind_turbine_component', '').lower()
    # Se procesan aquellos sensores que NO son parte del "tower"
    if component != 'tower':
        sensor_id = sp.get('id', 'desconocido')
        df_rotated = rotate_sensor_to_tower(sensor, datasets["Aventa"])
        if df_rotated is not None:
            rotated_dataframes[sensor_id] = df_rotated
            print(f"Sensor {sensor_id} rotado al marco de referencia de la torre.")

# 4. Ejemplo: visualizar y guardar el DataFrame resultante para un sensor, por ejemplo "GEN_01"
if "GEN_01" in rotated_dataframes:
    print(rotated_dataframes["GEN_01"].head())
    rotated_dataframes["GEN_01"].to_csv("GEN_01_rotated.csv", index=False)


No se encontró un dataset en el grupo 00_05_39.
No se encontró un dataset en el grupo 00_15_40.
No se encontró un dataset en el grupo 00_25_41.
No se encontró un dataset en el grupo 00_35_42.
No se encontró un dataset en el grupo 00_45_43.
No se encontró un dataset en el grupo 00_55_44.
No se encontró un dataset en el grupo 01_05_45.
No se encontró un dataset en el grupo 01_15_46.
No se encontró un dataset en el grupo 01_25_47.
No se encontró un dataset en el grupo 01_35_48.
No se encontró un dataset en el grupo 01_45_49.
No se encontró un dataset en el grupo 01_55_50.
No se encontró un dataset en el grupo 02_05_51.
No se encontró un dataset en el grupo 02_15_52.
No se encontró un dataset en el grupo 02_25_53.
No se encontró un dataset en el grupo 02_35_54.
No se encontró un dataset en el grupo 02_45_55.
No se encontró un dataset en el grupo 02_55_56.
No se encontró un dataset en el grupo 03_05_57.
No se encontró un dataset en el grupo 03_15_58.
No se encontró un dataset en el grupo 03

In [11]:
import json
import h5py
import pandas as pd
import numpy as np
from scipy.spatial.transform import Rotation

# 1. Leer el archivo de especificaciones de sensores.
with open("../../primer_modelo/Aventa_sensors.json", "r") as f:
    sensors_specs = json.load(f)

# 2. Leer el archivo HDF5 y organizarlo en un diccionario.
# Se asume que cada clave (excepto "ChannelList") es un canal con un dataset o, en caso de ser un grupo,
# se toma el primer dataset encontrado dentro de él. Se genera un vector de tiempo sintético.
datasets = {}
with h5py.File("../../primer_modelo/Aventa_Taggenberg_06_02_2022.hdf5", "r") as hdf:
    data_aventa = {}
    for channel in hdf.keys():
        if channel == "ChannelList":
            continue

        data_obj = hdf[channel]
        # Si es un dataset, lo leemos directamente
        if isinstance(data_obj, h5py.Dataset):
            data = data_obj[...]
        # Si es un grupo, se toma el primer dataset disponible dentro del grupo.
        elif isinstance(data_obj, h5py.Group):
            subkeys = list(data_obj.keys())
            if not subkeys:
                print(f"No se encontró un dataset en el grupo {channel}.")
                continue
            sub_obj = data_obj[subkeys[0]]
            if isinstance(sub_obj, h5py.Dataset):
                data = sub_obj[...]
            else:
                print(f"No se encontró un dataset en el grupo {channel}.")
                continue
        else:
            continue

        n_samples = len(data)
        # Vector de tiempo sintético: 0, 1, 2, ... n_samples-1
        time_vec = np.arange(n_samples)
        data_aventa[channel] = {"Time": time_vec, "Value": data}
    datasets["Aventa"] = data_aventa

def rotate_sensor_to_tower(sensor, dataset):
    """
    Rota los datos de un sensor (todos sus datos) al marco de referencia de la torre.

    Parámetros:
      sensor: diccionario con la especificación del sensor (debe incluir 'channels' y 'yaw-pitch-roll').
      dataset: diccionario con los datos medidos, donde las claves son los nombres de los canales.

    Retorna:
      DataFrame con las columnas "Time", "Xt", "Yt" y "Zt" (se descartan los datos originales).
    """
    channels = sensor.get('channels', [])
    
    # Verificar que el sensor tenga la propiedad de orientación definida
    if 'yaw-pitch-roll' not in sensor.get('sensor_placement', {}):
        print(f"Sensor {sensor['sensor_placement'].get('id', 'desconocido')} no tiene 'yaw-pitch-roll' definido; se omite la rotación.")
        return None
    ypr = sensor['sensor_placement']['yaw-pitch-roll']
    if any(val is None or str(val) == "None" for val in ypr):
        print(f"Sensor {sensor['sensor_placement'].get('id', 'desconocido')} tiene ángulos no definidos; se omite la rotación.")
        return None

    # Verificar que los tres canales estén definidos y existan en el dataset
    for ch in channels:
        if ch == "None" or ch not in dataset:
            print(f"Error al leer canal '{ch}' para sensor {sensor['sensor_placement'].get('id', 'desconocido')}.")
            return None

    # Crear el objeto de rotación con la convención 'ZYX'
    rotation = Rotation.from_euler('ZYX', ypr, degrees=True)
    
    # Leer los datos de cada canal
    try:
        X_data = dataset[channels[0]]["Value"]
        Y_data = dataset[channels[1]]["Value"]
        Z_data = dataset[channels[2]]["Value"]
        # Se utiliza el vector de tiempo del primer canal
        time_array = dataset[channels[0]]["Time"]
    except KeyError as e:
        print(f"Error al leer canales para sensor {sensor['sensor_placement'].get('id', 'desconocido')}: {e}")
        return None
    
    # Construir el DataFrame original
    df = pd.DataFrame({
        "Time": time_array,
        "Xs": X_data,
        "Ys": Y_data,
        "Zs": Z_data
    })
    
    # Aplicar la rotación a los datos locales
    data_local = df[['Xs', 'Ys', 'Zs']].to_numpy()
    data_rotated = rotation.apply(data_local)
    
    # Crear el DataFrame final (sólo con la columna de tiempo y los datos rotados)
    df_rotated = pd.DataFrame({
        "Time": df["Time"],
        "Xt": data_rotated[:, 0],
        "Yt": data_rotated[:, 1],
        "Zt": data_rotated[:, 2]
    })
    
    return df_rotated

# 3. Iterar sobre los sensores que no pertenecen al componente 'tower' y aplicar la rotación.
rotated_dataframes = {}

for sensor in sensors_specs['sensors']:
    component = sensor.get('sensor_placement', {}).get('wind_turbine_component', '').lower()
    # Se asume que los sensores que NO son parte del "tower" requieren rotación
    if component != 'tower':
        sensor_id = sensor.get('sensor_placement', {}).get('id', 'desconocido')
        df_rotated = rotate_sensor_to_tower(sensor, datasets["Aventa"])
        if df_rotated is not None:
            rotated_dataframes[sensor_id] = df_rotated
            print(f"Sensor {sensor_id} rotado al marco de referencia de la torre.")

# 4. Ejemplo: visualizar y guardar el DataFrame resultante para un sensor, por ejemplo "GEN_01"
if "GEN_01" in rotated_dataframes:
    print(rotated_dataframes["GEN_01"].head())
    rotated_dataframes["GEN_01"].to_csv("GEN_01_rotated.csv", index=False)


No se encontró un dataset en el grupo Aventa.
Error al leer canal 'None' para sensor NMF_01.
Error al leer canal 'NMF_ACC_XX_02' para sensor NMF_02.
Error al leer canal 'MSH_ACC_XX_01' para sensor MSH_01.
Error al leer canal 'MSH_ACC_XX_02' para sensor MSH_02.
Error al leer canal 'GEN_ACC_XX_01' para sensor GEN_01.
Sensor SCADA1 no tiene 'yaw-pitch-roll' definido; se omite la rotación.
Sensor SCADA2 no tiene 'yaw-pitch-roll' definido; se omite la rotación.
Sensor SCADA3 no tiene 'yaw-pitch-roll' definido; se omite la rotación.
Sensor SCADA4 no tiene 'yaw-pitch-roll' definido; se omite la rotación.
Sensor SCADA5 no tiene 'yaw-pitch-roll' definido; se omite la rotación.
